In [0]:
import pytest


silver_tables = [
    "silver_applicant_profiles",
    "silver_credit_applications",
    "silver_credit_history",
    "silver_economic_indicators",
    "silver_loan_details"
]


# ----------------------------------
# Test Silver Tables Exist
# ----------------------------------

@pytest.mark.parametrize("table", silver_tables)
def test_silver_table_exists(spark, table):

    df = spark.table(f"credits_catalog.silver_credit_risk_schema.{table}")

    assert df is not None
    assert df.count() >= 0


# ----------------------------------
# Test Data Quality Columns
# ----------------------------------

@pytest.mark.parametrize("table", silver_tables)
def test_data_quality_columns(spark, table):

    df = spark.table(f"credits_catalog.silver_credit_risk_schema.{table}")

    assert "data_quality_flag" in df.columns
    assert "error_message" in df.columns


# ----------------------------------
# Test Processing Timestamp
# ----------------------------------

@pytest.mark.parametrize("table", silver_tables)
def test_processing_timestamp_exists(spark, table):

    df = spark.table(f"credits_catalog.silver_credit_risk_schema.{table}")

    assert "processing_timestamp" in df.columns


# ----------------------------------
# Test No Duplicate Applicant IDs
# ----------------------------------

duplicate_tables = [
    "silver_applicant_profiles",
    "silver_credit_applications",
    "silver_credit_history",
    "silver_loan_details"
]


@pytest.mark.parametrize("table", duplicate_tables)
def test_no_duplicate_applicant_ids(spark, table):

    df = spark.table(f"credits_catalog.silver_credit_risk_schema.{table}")

    total = df.count()

    unique = df.select("applicant_id").distinct().count()

    assert total == unique


# ----------------------------------
# Test Data Quality Flags Valid
# ----------------------------------

@pytest.mark.parametrize("table", silver_tables)
def test_data_quality_values(spark, table):

    df = spark.table(f"credits_catalog.silver_credit_risk_schema.{table}")

    valid_flags = ["VALID", "INVALID_INCOME", "INVALID_RATE", "INVALID_LOAN",
                   "INVALID_SCORE", "INVALID_DTI", "INVALID_REGION",
                   "INVALID_YEAR", "INVALID_PROPERTY_VALUE",
                   "INVALID_INTEREST_RATE", "INVALID_TERM", "INVALID_LTV"]

    invalid_records = df.filter(~df.data_quality_flag.isin(valid_flags))

    assert invalid_records.count() == 0

In [0]:
%sh
pytest test_bronze_layer.py -v